# Epidemic curves from a JUNE2 events file (events-only)

Requires `requirements.txt` + `requirements-render.txt`.

## Parameters
Point `events_path` at any `simulation_events.h5`. The default is an example full-England run; swap it for the small committed example fixture (see ../README.md) -- the structure is identical.

In [39]:
import sys
from pathlib import Path

# Make the repo root importable so `core` resolves (no packaging yet).
repo_root = Path.cwd()
while not (repo_root / "core").is_dir() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# --- parameters ---
events_path = repo_root.parent / "JUNE2" / "runs" / "run_full_england_modern" / "simulation_events.h5"

print("repo_root :", repo_root)
print("events_path:", events_path, "(exists:", events_path.exists(), ")")

repo_root : /home/gavin/Documents/June_plus_plus/june_analysis
events_path: /home/gavin/Documents/June_plus_plus/JUNE2/runs/run_full_england_modern/simulation_events.h5 (exists: True )


## Inspect the events file
`inspect_file` lists every dataset in the file;

In [40]:
from core.load_data.june_events import inspect_file, load_enriched_events

summary = inspect_file(str(events_path))
for dataset in summary.datasets:
    print(dataset.path, dataset.n_rows, dataset.dtype)
print("registries:", list(summary.registries))

events/deaths 447 [('person_id', '<i4'), ('venue_id', '<i4'), ('time', '<f8')]
events/follows 209411097 {'names': ['host', 'follower', 'time', 'rule_id', 'slot'], 'formats': ['<i4', '<i4', '<f8', 'u1', '<i4'], 'offsets': [0, 4, 8, 16, 20], 'itemsize': 24}
events/hospital_admissions 2958 [('person_id', '<i4'), ('hospital_id', '<i4'), ('time', '<f8'), ('reason', 'S64')]
events/hospital_discharges 68 [('person_id', '<i4'), ('hospital_id', '<i4'), ('time', '<f8'), ('outcome', 'S64')]
events/icu_admissions 324 [('person_id', '<i4'), ('hospital_id', '<i4'), ('time', '<f8')]
events/infections 291026 {'names': ['person_id', 'infector_id', 'venue_id', 'time', 'encounter_type_id', 'transmission_mode_index', 'infector_symptom_id', 'source'], 'formats': ['<i4', '<i4', '<i4', '<f8', 'u1', 'u1', 'u1', 'u1'], 'offsets': [0, 4, 8, 16, 24, 25, 26, 27], 'itemsize': 32}
events/symptom_changes 196376 {'names': ['person_id', 'venue_id', 'time', 'old_symptom_id', 'new_symptom_id'], 'formats': ['<i4', '<i4',

### Auto-discover available event types
event types live under `events/`

In [41]:
available_event_types = sorted(
    dataset.path.split("/", 1)[1]
    for dataset in summary.datasets
    if dataset.path.startswith("events/") and dataset.n_rows > 0
)
print("available event types:", available_event_types)

available event types: ['deaths', 'follows', 'hospital_admissions', 'hospital_discharges', 'icu_admissions', 'infections', 'symptom_changes']


## Look at one events list

In [42]:
enriched_events = load_enriched_events(events_path, 'events/deaths')
enriched_events

,person_id,venue_id,time,person_age,person_sex,person_geo_unit_id,person_is_dead,person_death_time,person_schedule_type,person_num_activities,...,person_comorbidities,person_ethnicity,person_friendships,person_relationship_status,person_sexual_orientation,person_work_mode,venue_name,venue_type,venue_geo_unit_id,venue_n_subsets
0,49959490,22477915,4.503683,67.0,female,169041,0,-1.0,no_primary_activity,3,...,not applicable,B,[48225801 48820258 47962425],not applicable,heterosexual,not applicable,Venue_22477915,household,169041.0,3.0
1,45840107,2644561,13.820541,61.0,female,19793,0,-1.0,no_primary_activity,3,...,not applicable,W,[45481121 42316860 49701684],not applicable,heterosexual,Normal,Venue_2644561,household,19793.0,1.0
2,42309636,2198503,18.773343,56.0,male,16467,0,-1.0,no_primary_activity,3,...,not applicable,A,[40284317 36207183 48870375],not applicable,heterosexual,From_Home,Venue_2198503,household,16467.0,1.0
3,57332454,2420170,22.569357,82.0,female,18157,0,-1.0,no_primary_activity,3,...,not applicable,W,[58968873 54439466 53053089],not applicable,heterosexual,not applicable,Venue_2420170,household,18157.0,3.0
4,54723079,2590947,24.140769,75.0,male,19389,0,-1.0,no_primary_activity,3,...,not applicable,W,[56103414 54232894 53964231],not applicable,heterosexual,not applicable,Venue_2590947,household,19389.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
442,54991444,7458395,21.323103,76.0,female,56036,0,-1.0,no_primary_activity,3,...,not applicable,W,[45911273 57199221 58356599],not applicable,heterosexual,not applicable,Venue_7458395,household,56036.0,1.0
443,59126184,4665946,25.619872,93.0,female,34908,0,-1.0,no_primary_activity,3,...,not applicable,W,[56656754 57693099 58753711],not applicable,heterosexual,not applicable,Venue_4665946,household,34908.0,1.0
444,54531426,25699451,25.607966,75.0,female,63021,0,-1.0,no_primary_activity,3,...,not applicable,W,[46669092 56684685 56857679],not applicable,heterosexual,not applicable,Venue_25699451,cinema,190457.0,1.0
445,53077929,25976149,29.261336,72.0,male,35270,0,-1.0,no_primary_activity,3,...,not applicable,W,[57693434 48005756 48005757],not applicable,heterosexual,not applicable,Venue_25976149,cinema,191382.0,1.0


## Load, aggregate, build curves
For each chosen event type: load the **enriched** table (events + lookup joins, so each row already carries `geo_unit_id`). Group by day to get curves of events/day

In [44]:
event_types = ["infections", "deaths", 'hospital_admissions']  # which curves to plot

curves = {}
for event_type in event_types:
    if event_type not in available_event_types:
        print(f"skipping {event_type!r} -- not present in this file")
        continue
    enriched_events = load_enriched_events(str(events_path), f"events/{event_type}")
    
    curves[event_type] = enriched_events.groupby(enriched_events["time"].astype(int)).size()

    print(f"{event_type}: {int(curves[event_type].sum())} events ")


infections: 291026 events 
deaths: 447 events 
hospital_admissions: 2958 events 


## Plot

Note: The first day start with the infection seeds, so there might be an unexpected dip in the first day or two. 

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 8), sharex=True)

# Only plot curves the file actually produced (the build loop skips absent types).
if "infections" in curves:
    ax1.plot(curves["infections"].index, curves["infections"], label="infections")
ax1.set_xlabel("time (days)")
ax1.set_ylabel("events per day")
ax1.set_title("Epidemic curves (events-only)")
ax1.set_ylim(bottom=0)

if "deaths" in curves:
    ax2.plot(curves["deaths"].index, curves["deaths"],
             label="deaths", color="black", linestyle="dotted")
if "hospital_admissions" in curves:
    ax2.plot(curves["hospital_admissions"].index, curves["hospital_admissions"],
             label="hospital_admissions", color="forestgreen", linestyle="dashed")
ax2.set_xlabel("time (days)")
ax2.set_ylabel("events per day")
ax2.set_ylim(bottom=0)

ax1.legend()
ax2.legend()
fig.tight_layout()
plt.show()

In [ ]:
from core.load_data import load_geo_events
from core.aggregate import aggregate_events, to_long_dataframe

days_per_bin = 1.0

# Per-geo aggregate via the light geo path: load_geo_events resolves one
# geo_unit_id per event (joining only that lookup column, not the full people/venue
# metadata of load_enriched_events), then aggregate_events bins time x geo.
if "infections" in available_event_types:
    located_infections = load_geo_events(str(events_path), "events/infections")
    infections_aggregate = aggregate_events(
        located_infections, event_type="infections", days_per_bin=days_per_bin
    )
    long_frame = to_long_dataframe(infections_aggregate)
    long_frame.head()